# Testing Pipeline on Cholecystectomy Video

This notebook downloads a real laparoscopic cholecystectomy video from YouTube and runs the full pipeline on it. Add these cells to your existing Colab notebook AFTER Step 3 (project upload), replacing the sample video download.

## Prerequisites

You should have already:
1. GPU enabled (T4)
2. Dependencies installed (from previous notebook)
3. Your project uploaded and extracted
4. Working directory changed to project folder

## Option 1: Download from YouTube (Fastest)

This downloads a real laparoscopic cholecystectomy video for testing.

In [ ]:
# Install yt-dlp for YouTube downloading
!pip install -q yt-dlp

In [ ]:
import os
os.makedirs('videos', exist_ok=True)

# Search YouTube yourself for 'laparoscopic cholecystectomy' and paste a URL here.
# Look for videos that are:
# - 5-15 minutes long (shorter = faster testing)
# - Clear laparoscopic camera footage (not lecture/interview)
# - Good video quality (720p or better)
#
# EXAMPLE search results that typically work well:
# - Educational surgery channels
# - Medical school teaching videos
# - Surgeon-uploaded procedure demonstrations

youtube_url = 'PASTE_YOUR_YOUTUBE_URL_HERE'

if youtube_url == 'PASTE_YOUR_YOUTUBE_URL_HERE':
    print('ERROR: Replace youtube_url with an actual URL first!')
    print('\nGo to YouTube, search "laparoscopic cholecystectomy", pick a video, copy its URL.')
else:
    !yt-dlp -f 'best[height<=720][ext=mp4]/best[height<=720]' -o 'videos/surgery.mp4' {youtube_url}
    
    if os.path.exists('videos/surgery.mp4'):
        size = os.path.getsize('videos/surgery.mp4') / 1024 / 1024
        print(f'\nDownloaded: videos/surgery.mp4 ({size:.1f} MB)')
    else:
        print('Download failed. Try a different video URL.')

In [ ]:
# Verify video properties
import cv2

video_path = 'videos/surgery.mp4'
cap = cv2.VideoCapture(video_path)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = total_frames / fps if fps > 0 else 0
cap.release()

print(f'Video: {video_path}')
print(f'  Resolution: {width}x{height}')
print(f'  Total frames: {total_frames}')
print(f'  FPS: {fps:.1f}')
print(f'  Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)')

## Preview a few frames before running the pipeline

Sanity check that the video looks like actual surgical footage before spending time on it.

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Sample 4 frames evenly across the video
sample_indices = [total // 5, total // 3, total // 2, 2 * total // 3]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, frame_idx in enumerate(sample_indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if ret:
        ax = axes[i // 2, i % 2]
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {frame_idx} (t={frame_idx/30:.1f}s)')
        ax.axis('off')

cap.release()
plt.tight_layout()
plt.show()

print('\nWhat to look for:')
print('- Video should show inside the body (laparoscopic view)')
print('- You should see instruments (graspers, scissors, cautery)')
print('- Colors are typically pink/red tissue with blueish gray tools')
print('- If you only see people talking or diagrams, get a different video')

## Configure the pipeline for cholecystectomy

Tune detection prompts specifically for the instruments used in gallbladder surgery.

In [ ]:
import yaml

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Cholecystectomy-specific instrument prompts
# These are the actual instruments in Cholec80 dataset
config['detection']['prompts'] = [
    'grasper',
    'bipolar forceps',
    'hook electrocautery',
    'scissors',
    'clip applier',
    'irrigator suction',
    'specimen bag',
    'surgical instrument',  # catchall for anything missed
]

# For first test, limit frames to see results quickly
config['video']['max_frames'] = 100

# Use adaptive sampling (skip boring frames)
config['temporal_sampling']['mode'] = 'adaptive'

# Enable video propagation (the whole point of using Colab)
config['video_propagation']['enabled'] = True

# Smaller model for faster inference on free T4 GPU
config['detection']['model_id'] = 'IDEA-Research/grounding-dino-tiny'

# Lower threshold since surgical footage is tricky
config['detection']['box_threshold'] = 0.25
config['detection']['text_threshold'] = 0.20

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config updated for cholecystectomy testing:')
print(f'  Detection prompts: {config["detection"]["prompts"]}')
print(f'  Max frames: {config["video"]["max_frames"]}')
print(f'  Box threshold: {config["detection"]["box_threshold"]}')
print(f'  Video propagation: {config["video_propagation"]["enabled"]}')

## Run the pipeline

First run will download model weights (~2GB), so it takes 5-10 minutes.
Subsequent runs are much faster.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.pipeline import SurgicalAnnotationPipeline

pipeline = SurgicalAnnotationPipeline('config.yaml')
results = pipeline.run('videos/surgery.mp4')

In [ ]:
# View statistics
import json
print(json.dumps(results['stats'], indent=2))

## View sample results

Look at a few annotated frames to see what the pipeline detected.

In [ ]:
from IPython.display import Image, display
import os
import glob

# Find visualization directory
viz_dirs = glob.glob('output/**/visualizations', recursive=True)
if viz_dirs:
    viz_dir = viz_dirs[0]
    viz_files = sorted(os.listdir(viz_dir))
    
    # Sample 5 frames evenly
    if len(viz_files) > 5:
        step = len(viz_files) // 5
        viz_files = viz_files[::step][:5]
    
    for f in viz_files:
        print(f'\n=== {f} ===')
        display(Image(os.path.join(viz_dir, f), width=800))
else:
    print('No visualizations found. Check output folder:')
    !ls -la output/

## Create annotated video

Combine annotated frames into a video for the demo.

In [ ]:
import cv2
import os
import glob

viz_dirs = glob.glob('output/**/visualizations', recursive=True)
viz_dir = viz_dirs[0] if viz_dirs else None

if viz_dir:
    output_path = 'output/annotated_demo.mp4'
    frame_paths = sorted(glob.glob(os.path.join(viz_dir, '*.jpg')))
    
    if frame_paths:
        first = cv2.imread(frame_paths[0])
        h, w = first.shape[:2]
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(output_path, fourcc, 5, (w, h))
        
        for path in frame_paths:
            writer.write(cv2.imread(path))
        writer.release()
        
        # Convert for browser playback
        !ffmpeg -i {output_path} -vcodec libx264 -acodec aac -y output/annotated_web.mp4 2>/dev/null
        print(f'Video ready: output/annotated_web.mp4')
        print(f'Frames: {len(frame_paths)}, Duration: {len(frame_paths)/5:.1f}s at 5fps playback')

In [ ]:
# Play video inline
from IPython.display import HTML
from base64 import b64encode

with open('output/annotated_web.mp4', 'rb') as f:
    video_data = f.read()

video_b64 = b64encode(video_data).decode()

HTML(f'''
<video width="800" controls autoplay loop>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
''')

## Review the results honestly

**What should look good:**
- Instruments detected consistently across frames
- Tracking IDs stay the same for the same tool across time
- Segmentation masks tightly follow instrument shapes

**What might not look great (expected):**
- False positives on tissue (looks like something else to the model)
- Missed detections when instruments are partially hidden
- Confusion between similar-looking tools (grasper vs bipolar forceps)

**Improvement strategies to try:**

1. **Refine prompts:** Add or remove specific instruments based on what's in the video
2. **Adjust thresholds:** Increase box_threshold if too many false positives, decrease if missing detections
3. **Try larger model:** Change to 'IDEA-Research/grounding-dino-base' for better accuracy (slower)
4. **More frames:** Increase max_frames for longer sequences (video propagation gets better with more context)

## Option 2: Using Cholec80 Dataset (After Approval)

Once you get access to Cholec80 (via camma.u-strasbg.fr registration), you can use it for proper evaluation with ground truth.

The dataset comes as .mp4 files plus tool annotation TXT files. Here's how to use them:

In [ ]:
# After downloading Cholec80, upload one video to /content/videos/
# Example filename: video01.mp4

# The dataset uses these 7 instruments:
config['detection']['prompts'] = [
    'grasper',
    'bipolar',
    'hook',
    'scissors',
    'clipper',
    'irrigator',
    'specimen bag',
]

# Run pipeline
results = pipeline.run('videos/video01.mp4')

In [ ]:
# To evaluate against ground truth (after converting Cholec80 annotations to COCO):
# results = pipeline.run('videos/video01.mp4', ground_truth_path='cholec80_gt.json')
# print(results['evaluation'])

# Note: Cholec80 comes with per-frame tool presence annotations (binary flags),
# not bounding boxes. For bounding box evaluation you'd need CholecT50 or manual conversion.

## Download results to your computer

In [ ]:
!zip -r cholec_results.zip output/

from google.colab import files
files.download('cholec_results.zip')